In [47]:
import glob, json, os, sys

sys.path.append('../swebench/metrics')
from conversion import convert_log_to_ground_truth
from getters import get_logs_gold
from monitor import monitor_validation, monitor_logs_same_diff
sys.path = sys.path[:-1]

sys.path.append('../swebench/harness')
sys.path = sys.path[:-1]

sys.path.append('../swebench/collect')
from convert_to_dataset import convert_to_hf_dataset

Configure Repo, logs, output directories (REQUIRED)

In [38]:
repo = 'ankidroid/Anki-Android' # TODO: Replace with github repository name (e.g. 'Kotlin/kotlinx-datetime')
log_dir = '../validation_logs/run_20250328_235815' # TODO: Replace with path to folder of execution logs
tasks = json.load(open('../task-instances/versioned/Anki-Android-task-instances_versions.json')) # TODO: Replace with path to versioned candidate task instances

include_errored_tests = True # TODO Choose whether to include errored tests in the validated task instances list
suffix = "with_errored_tests" if include_errored_tests else "without_errored_tests"

# TODO: Replace this with a path to a .json file to save the task instances
FINAL_VALIDATED_TASK_INSTANCES = f"../task-instances/validated/Anki-Android-task-instances-{suffix}.json" 

hf_dataset_dir = f"../datasets/anki-android-test/base-{suffix}" # TODO: Replace with path to output HuggingFace Dataset Directory



Get map of version to setup commit

In [39]:
tasks = sorted(tasks, key=lambda x: x['created_at'], reverse=True)
version_to_setup_commit = {}
for t in tasks:
    if 'version' in t and t['version'] not in version_to_setup_commit:
        version_to_setup_commit[t['version']] = t['base_commit']
assert(
    sorted(list([x or "" for x in version_to_setup_commit.keys()])) ==
    sorted(list(set([t['version'] or "" for t in tasks if 'version' in t])))
)
tasks = {t['instance_id']: t for t in tasks}

#### Monitor Validation

In [40]:
failed_install, corrupt_test_patch, corrupt_patch, timeout, success = monitor_validation(log_dir)

Total Attempts: 164
Failed: 0
Usable: 164
Corrupt Test: 2
Corrupt Diff: 9
Test Script Timeout: 0
Success E2E: 153


In [41]:
logs_same, logs_diff = monitor_logs_same_diff(log_dir, repo)
print(f"Logs same: {len(logs_same)}")
print(f"Logs diff: {len(logs_diff)}")

Logs same: 65
Logs diff: 88


#### Get [FP]2[FP] Tests

In [42]:
inst_id_to_gt = {}
for d in logs_diff:
    status_gt = convert_log_to_ground_truth(d[0], repo)
    inst_id_to_gt[d[0]] = status_gt

# Aggregate counts for each test status across all instances
status_counts = {}
for ground_truth in inst_id_to_gt.values():
    for test_status, tests in ground_truth.items():
        if test_status not in status_counts:
            status_counts[test_status] = 0
        status_counts[test_status] += len(tests)

print("Aggregated Test Status Counts:")
for test_status, count in status_counts.items():
    if count > 0:  # Only print statuses with non-zero counts
        print(f"  {test_status}: {count}")


Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converting log to ground truth
Converti

#### Create Task Instances `.json` file

In [43]:
tasks_final = []
get_id_from_log = lambda x: x.split('/')[-1].rsplit('.', 1)[0]
for k, v in inst_id_to_gt.items():
    # Filtering for tasks that failed then passed
    # TODO: Only consider tasks that failed then passed
    if len(v['FAIL_TO_PASS']) == 0 and (include_errored_tests and len(v['ERROR_TO_PASS'])) == 0:
        continue
    
    task = tasks[get_id_from_log(k)]
    task['FAIL_TO_PASS'] = v['FAIL_TO_PASS']
    task['PASS_TO_PASS'] = v['PASS_TO_PASS']
    task['ERROR_TO_PASS'] = v['ERROR_TO_PASS']
    task['environment_setup_commit'] = version_to_setup_commit[task['version']]

    # Do not consider tasks where the log before the patch has an attribute/import error
    log_path = os.path.join(log_dir, f'{task["instance_id"]}.log')
    log_before, log_after = get_logs_gold(log_path)

    tasks_final.append(task)
print(f"Final number of task instances: ", len(tasks_final))
print("\nTask Instance IDs:")
for task in sorted(tasks_final, key=lambda x: x["instance_id"]):
    print(task["instance_id"])

with open(FINAL_VALIDATED_TASK_INSTANCES, 'w') as f:
    json.dump(tasks_final, fp=f)

Final number of task instances:  88

Task Instance IDs:
ankidroid__Anki-Android-14114
ankidroid__Anki-Android-14130
ankidroid__Anki-Android-14182
ankidroid__Anki-Android-14230
ankidroid__Anki-Android-14291
ankidroid__Anki-Android-14360
ankidroid__Anki-Android-14362
ankidroid__Anki-Android-14369
ankidroid__Anki-Android-14388
ankidroid__Anki-Android-14483
ankidroid__Anki-Android-14564
ankidroid__Anki-Android-14622
ankidroid__Anki-Android-14652
ankidroid__Anki-Android-14719
ankidroid__Anki-Android-14738
ankidroid__Anki-Android-14740
ankidroid__Anki-Android-14769
ankidroid__Anki-Android-14859
ankidroid__Anki-Android-14959
ankidroid__Anki-Android-14967
ankidroid__Anki-Android-14987
ankidroid__Anki-Android-14990
ankidroid__Anki-Android-15000
ankidroid__Anki-Android-15010
ankidroid__Anki-Android-15055
ankidroid__Anki-Android-15076
ankidroid__Anki-Android-15141
ankidroid__Anki-Android-15251
ankidroid__Anki-Android-15252
ankidroid__Anki-Android-15264
ankidroid__Anki-Android-15268
ankidroid__Ank

#### Convert Task Instances JSON file to a HuggingFace Dataset Directory (Required)

In [48]:
import json

# Read the JSON file
with open(FINAL_VALIDATED_TASK_INSTANCES, 'r') as f:
    task_instances = json.load(f)  # Load the JSON data

# Now call the function with the loaded data
dataset = convert_to_hf_dataset(task_instances, hf_dataset_dir)

# Preview a few examples
print(dataset['test'][:2])

Saving the dataset (1/1 shards): 100%|██████████| 88/88 [00:00<00:00, 21111.87 examples/s]

Dataset saved to ../datasets/anki-android-test/base-with_errored_tests
{'repo': ['ankidroid/Anki-Android', 'ankidroid/Anki-Android'], 'pull_number': [14990, 15287], 'instance_id': ['ankidroid__Anki-Android-14990', 'ankidroid__Anki-Android-15287'], 'issue_numbers': [['14989'], ['13795']], 'base_commit': ['121711e900117d968a65bd57c30f4bc774bcddfe', '93a1bea370cf738db6029106814565264eb4b698'], 'patch': ['diff --git a/AnkiDroid/src/main/java/com/ichi2/anki/CardBrowser.kt b/AnkiDroid/src/main/java/com/ichi2/anki/CardBrowser.kt\nindex ac34c1064fdf..4e83cca15072 100644\n--- a/AnkiDroid/src/main/java/com/ichi2/anki/CardBrowser.kt\n+++ b/AnkiDroid/src/main/java/com/ichi2/anki/CardBrowser.kt\n@@ -45,6 +45,7 @@ import com.ichi2.anki.CollectionManager.withCol\n import com.ichi2.anki.Previewer.Companion.toIntent\n import com.ichi2.anki.UIUtils.showThemedToast\n import com.ichi2.anki.browser.CardBrowserViewModel\n+import com.ichi2.anki.browser.SaveSearchResult\n import com.ichi2.anki.dialogs.*\n imp